In [49]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from scipy.special import beta, gammaln, hyp2f1

In [50]:
# каталог с ноутбуками и корень проекта
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
from src.analytics import assess, guess_mapping, normalize, read_table
from src.analytics.quality import drop_duplicates, trim_incomplete_tail
from src.analytics.schema import AMOUNT, CUSTOMER_ID, ORDER_DATE
# данные
DATA_PATH = ROOT / "data" / "raw" / "online_retail_II.xlsx"
raw = read_table(DATA_PATH)
source = "Online Retail II"
# распознавание колонок
mapping = guess_mapping(raw)

for canonical, column in mapping.as_dict().items():
    print(f"{canonical:<12} <- {column}")

if mapping.unmapped:
    print("\nНе распознано:", ", ".join(map(str, mapping.unmapped)))

assert mapping.is_valid, f"Не хватает колонок: {mapping.missing_required}"
# отчистка 
clean = normalize(raw, mapping)
clean, duplicates = drop_duplicates(clean)
clean, tail_trimmed = trim_incomplete_tail(clean)
clean.head()

order_date   <- InvoiceDate
customer_id  <- Customer ID
order_id     <- Invoice
quantity     <- Quantity
unit_price   <- Price

Не распознано: StockCode, Description, Country


,order_date,customer_id,amount,order_id,quantity,unit_price
0,2009-12-01 07:45:00,13085,83.4,489434,12.0,6.95
1,2009-12-01 07:45:00,13085,81.0,489434,12.0,6.75
2,2009-12-01 07:45:00,13085,100.8,489434,48.0,2.10
3,2009-12-01 07:45:00,13085,30.0,489434,24.0,1.25
4,2009-12-01 07:45:00,13085,39.6,489434,24.0,1.65


In [ ]:
class BGNBDFitter:

    def __init__(self):
        self.params = None  # [r, alpha, a, b]

    @staticmethod
    def _negative_log_likelihood(params, frequency, recency, T):
        """Отрицательный Log-Likelihood для BG/NBD."""
        r, alpha, a, b = params
        if any(p <= 0 for p in params):
            return np.inf

        x = np.array(frequency)
        t_x = np.array(recency)
        T = np.array(T)

        # Вычисляем общие части
        A1 = gammaln(r + x) - gammaln(r) + r * np.log(alpha)
        A2 = gammaln(a + b) + gammaln(b + x) - gammaln(b) - gammaln(a + b + x)

        log_term1 = -((r + x) * np.log(alpha + T))

        idx = x > 0
        ll = np.zeros_like(x, dtype=float)

        # x == 0
        ll[~idx] = A1[~idx] + r * np.log(alpha / (alpha + T[~idx]))

        # x > 0
        if np.any(idx):
            log_term2 = (
                np.log(a)
                - np.log(b + x[idx] - 1)
                - (r + x[idx]) * np.log(alpha + t_x[idx])
            )
            max_term = np.maximum(log_term1[idx], log_term2)
            combined_term = max_term + np.log(
                np.exp(log_term1[idx] - max_term)
                + np.exp(log_term2 - max_term)
            )
            ll[idx] = A1[idx] + A2[idx] + combined_term

        # Защита от NaN/Inf в сумме
        total_ll = -np.sum(ll)
        return total_ll if np.isfinite(total_ll) else np.inf

    def fit(self, frequency, recency, T):
        """Обучение модели методом L-BFGS-B с расширенными лимитами итераций."""
        # Более надежная стартовая точка для данных в масштабе дней
        initial_guess = np.array([0.5, 5.0, 0.5, 5.0])

        # Ограничения: параметры не должны падать в ноль
        bounds = [(0.005, 100.0), (0.005, 200.0), (0.005, 100.0), (0.005, 200.0)]

        res = minimize(
            self._negative_log_likelihood,
            initial_guess,
            args=(np.array(frequency), np.array(recency), np.array(T)),
            bounds=bounds,
            method="L-BFGS-B",
            options={
                "maxiter": 2000,
                "ftol": 1e-5,
            },  
        )

        if not res.success:
            print(
                "Предупреждение: оптимизация не сошлась, используем лучшие найденные параметры.",
                res.message,
            )

        self.params = res.x
        return self

    def predict_p_alive(self, frequency, recency, T):
        r, alpha, a, b = self.params
        x = np.array(frequency)
        t_x = np.array(recency)
        T = np.array(T)

        term1 = (a / (b + x - 1)) * ((alpha + T) / (alpha + t_x)) ** (r + x)
        p_alive = 1.0 / (1.0 + (x > 0) * term1)
        return np.clip(p_alive, 0, 1)

    def predict_expected_transactions(self, t, frequency, recency, T):
        r, alpha, a, b = self.params
        x = np.array(frequency)
        t_x = np.array(recency)
        T = np.array(T)

        p_alive = self.predict_p_alive(x, t_x, T)
        expected_tx = (
            p_alive
            * (r + x)
            * t
            / (alpha + T)
            * (b / (a + b + x - 1))
        )
        return expected_tx


In [52]:
# Модель Gamma-Gamma
class GammaGammaFitter:

    def __init__(self):
        self.params = None  # [p, q, v]

    @staticmethod
    def _negative_log_likelihood(params, frequency, monetary_value):
        """Отрицательный Log-Likelihood для Gamma-Gamma"""
        p, q, v = params
        if any(param <= 0 for param in params):
            return np.inf

        x = frequency
        m = monetary_value

        # Формула плотности распределения Gamma-Gamma в логарифмическом виде
        ll = (
            gammaln(p * x + q)
            - gammaln(p * x)
            - gammaln(q)
            + q * np.log(v)
            + p * x * np.log(x * m)
            - (p * x + q) * np.log(v + x * m)
        )
        return -np.sum(ll)

    def fit(self, frequency, monetary_value):
        """Обучение модели среднего чека"""
        # Gamma-Gamma обучается только на клиентах С ПОВТОРНЫМИ покупками (frequency > 0)
        idx = frequency > 0
        x = np.array(frequency)[idx]
        m = np.array(monetary_value)[idx]

        initial_guess = np.array([1.0, 1.0, 1.0])
        bounds = [(0.001, None), (0.001, None), (0.001, None)]

        res = minimize(
            self._negative_log_likelihood,
            initial_guess,
            args=(x, m),
            bounds=bounds,
            method="L-BFGS-B",
        )

        self.params = res.x
        return self

    def predict_conditional_expected_average_profit(
        self, frequency, monetary_value
    ):
        """Вычисляет ожидаемый средний чек с учетом индивидуальной истории"""
        p, q, v = self.params
        x = np.array(frequency)
        m = np.array(monetary_value)

        # Если у клиента 0 повторных покупок, его лучшая оценка — это среднее по базе (v * p / (q - 1))
        # Иначе применяется байесовское обновление
        individual_weight = (p * x) / (p * x + q - 1)
        expected_avg_profit = (
            individual_weight * m + (1 - individual_weight) * (v * p) / (q - 1)
        )

        # Для клиентов без истории возвращаем базовое среднее распределения
        base_mean = (v * p) / (q - 1)
        return np.where(x == 0, base_mean, expected_avg_profit)

In [59]:
df = clean.copy()
df = df[df['amount'] > 0]
# разделяем данные на train и test
min_date = pd.to_datetime(df['order_date']).min()
split_date = min_date + pd.DateOffset(months=9)

train_transactions = df[pd.to_datetime(df['order_date']) < split_date].copy()
test_transactions = df[pd.to_datetime(df['order_date']) >= split_date].copy() 

snapshot_train = train_transactions['order_date'].max()
rfmt_train = train_transactions.groupby('customer_id').agg(
    _first_pur=('order_date', 'min'),
    _last_pur=('order_date', 'max'),
    _pur_days=('order_date', 'nunique'),
    Monetary=('amount', 'mean')
).reset_index()

rfmt_train['Recency'] = (rfmt_train['_last_pur'] - rfmt_train['_first_pur']).dt.days
rfmt_train['Frequency'] = rfmt_train['_pur_days'] - 1
rfmt_train['T'] = (snapshot_train - rfmt_train['_first_pur']).dt.days

rfmt_train = rfmt_train[['customer_id', 'Recency', 'Frequency', 'Monetary', 'T']]
print("Размер обучающей выборки клиентов:", rfmt_train.shape[0])
print(rfmt_train)

Размер обучающей выборки клиентов: 3321
     customer_id  Recency  Frequency   Monetary    T
0          12346      196         10  18.349167  260
1          12349       18          1  31.033548  124
2          12355        0          0  22.383571  102
3          12358      181          1  37.817917  266
4          12359      198          4  25.206129  269
...          ...      ...        ...        ...  ...
3316       18281        0          0  13.391429  112
3317       18283      180          3   3.500290  193
3318       18285        0          0  35.583333  195
3319       18286      247          1  21.312368  258
3320       18287        0          0  21.592963  106

[3321 rows x 5 columns]


In [60]:
# Инициализируем и обучаем модель BG/NBD (число покупок)
bgnbd = BGNBDFitter()
bgnbd.fit(rfmt_train["Frequency"], rfmt_train["Recency"], rfmt_train["T"])
print("Параметры BG/NBD (r, alpha, a, b):", bgnbd.params)

Параметры BG/NBD (r, alpha, a, b): [100.         200.          46.27857857 151.55715001]


In [61]:
# Инициализируем и обучаем модель Gamma-Gamma(средний чек)
gg = GammaGammaFitter()
gg.fit(rfmt_train["Frequency"], rfmt_train["Monetary"])
print("Параметры Gamma-Gamma (p, q, v):", gg.params)

Параметры Gamma-Gamma (p, q, v): [11.80355234  2.11737039  2.96507955]


In [62]:
# Считаем промежуточные вероятности и чеки для каждого клиента
rfmt_train["P_alive"] = bgnbd.predict_p_alive(
    rfmt_train["Frequency"], rfmt_train["Recency"], rfmt_train["T"]
)
rfmt_train["Expected_Monetary"] = gg.predict_conditional_expected_average_profit(
    rfmt_train["Frequency"], rfmt_train["Monetary"]
)

for days in [90, 180, 365]:
    # Ожидаемое количество транзакций за период дней
    exp_tx = bgnbd.predict_expected_transactions(
        days, rfmt_train["Frequency"], rfmt_train["Recency"], rfmt_train["T"]
    )
    # Перемножаем транзакции на средний чек -> получаем прогнозный CLV
    rfmt_train[f"CLV_{days}"] = exp_tx * rfmt_train["Expected_Monetary"]

print("\nПрогнозы CLV успешно рассчитаны!")
print(
    rfmt_train[
        [
            "customer_id",
            "P_alive",
            "Expected_Monetary",
            "CLV_90",
            "CLV_180",
            "CLV_365",
        ]
    ].head()
)


Прогнозы CLV успешно рассчитаны!
  customer_id       P_alive  Expected_Monetary        CLV_90       CLV_180  \
0       12346  2.417456e-07          18.470823  7.041626e-05  1.408325e-04   
1       12349  1.362248e-17          31.058508  9.093423e-15  1.818685e-14   
2       12355  1.000000e+00          31.322176  7.187204e+02  1.437441e+03   
3       12358  4.804425e-09          37.256181  2.674788e-06  5.349576e-06   
4       12359  1.286983e-07          25.347525  4.913002e-05  9.826003e-05   

        CLV_365  
0  2.855771e-04  
1  3.687888e-14  
2  2.914811e+03  
3  1.084775e-05  
4  1.992495e-04  


In [63]:
# Собираем РЕАЛЬНУЮ выручку за тестовый период (последние 3 месяца)
rfmt_test_actual = (
    test_transactions.groupby("customer_id")
    .agg(Actual_Revenue=("amount", "sum"))
    .reset_index()
)

#  Объединяем наш прогноз CLV_90 и реальный факт по каждому клиенту
validation_df = pd.merge(
    rfmt_train[["customer_id", "CLV_90"]],
    rfmt_test_actual,
    on="customer_id",
    how="left",
)

# Если клиента не было в тестовом периоде, значит его фактическая выручка = 0
validation_df["Actual_Revenue"] = validation_df["Actual_Revenue"].fillna(0)

# Расчет метрик качества по вашему ТЗ
# MAE на уровне отдельного клиента
mae = np.mean(np.abs(validation_df["CLV_90"] - validation_df["Actual_Revenue"]))

# Смещение (Bias) на уровне всей базы (общий прогноз базы vs общий факт базы)
total_predicted = validation_df["CLV_90"].sum()
total_actual = validation_df["Actual_Revenue"].sum()
base_bias = total_predicted - total_actual
bias_percentage = (base_bias / total_actual) * 100

# Вывод итогов
print("=== РЕЗУЛЬТАТЫ ВАЛИДАЦИИ МОДЕЛИ (ГОРИЗОНТ 90 ДНЕЙ) ===")
print(f"MAE на уровне клиента: {mae:.2f}")
print(f"Общий прогноз всей базы: {total_predicted:.2f}")
print(f"Общий факт всей базы:    {total_actual:.2f}")
print(f"Смещение базы (Bias):    {base_bias:.2f} ({bias_percentage:+.2f}%)")

=== РЕЗУЛЬТАТЫ ВАЛИДАЦИИ МОДЕЛИ (ГОРИЗОНТ 90 ДНЕЙ) ===
MAE на уровне клиента: 757.53
Общий прогноз всей базы: 981433.09
Общий факт всей базы:    2041397.00
Смещение базы (Bias):    -1059963.92 (-51.92%)


In [64]:
# Топ 10 ценных клиентов по результатам прогноза CLV
top_valuable_customers = rfmt_train.sort_values(by='CLV_365', ascending=False).head(10)
print("=== ТОП-10 САМЫХ ЦЕННЫХ КЛИЕНТОВ БАЗЫ (ПРОГНОЗ НА 365 ДНЕЙ) ===")

print(top_valuable_customers[[
    'customer_id', 
    'Frequency', 
    'P_alive', 
    'Expected_Monetary', 
    'CLV_90', 
    'CLV_365'
]].to_string(index=False, formatters={
    'P_alive': '{:.4f}'.format,
    'Expected_Monetary': '{:.2f}'.format,
    'CLV_90': '{:.2f}'.format,
    'CLV_365': '{:.2f}'.format
}))

=== ТОП-10 САМЫХ ЦЕННЫХ КЛИЕНТОВ БАЗЫ (ПРОГНОЗ НА 365 ДНЕЙ) ===
customer_id  Frequency P_alive Expected_Monetary  CLV_90  CLV_365
      18102         74  0.8291            442.16 6792.03 27545.47
      17949         55  0.3794            638.25 4307.47 17469.19
      14045          9  0.7752            294.93 3796.21 15395.74
      17553          2  0.2895            543.27 3189.44 12934.96
      13848          3  0.3571            349.95 1889.27  7662.04
      12931         25  0.3704            261.69 1575.15  6388.09
      16656          9  0.7752            130.55 1555.23  6307.31
      17133          4  0.1970            344.97 1322.38  5363.00
      13027          9  0.3587            195.95 1200.13  4867.19
      12875          2  0.4418            162.74 1164.64  4723.25
